In [0]:
%sql
use catalog trueanalytics_data;

In [0]:
import pyspark
import pyspark.sql.functions as F
import pyspark.sql.types as T 
from functools import partial
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

In [0]:
def save_to_parquet(df, save_path):
    (df.write.format('parquet')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save parquet to ", save_path)

def save_to_csv(df, save_path):
    (df.coalesce(1)
        .write.format('csv')
        .mode('overwrite')
        .option("header", "true")
        .save(save_path)
    )
    return print("successfully save csv to ", save_path)

In [0]:
# parameter: par_month
dbutils.widgets.text("par_month", "202510")
par_month = dbutils.widgets.get("par_month")

try:
  par_month = int(par_month)
except ValueError:
  par_month = 0
  raise ValueError("par_month value must be numeric")

if par_month!=0:
  pass
else:
  dbutils.notebook.exit("Aborting as ondition not met. Further tasks will be skipped")

# debug
display(par_month)

In [0]:
# master data
prep_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/{par_month}_footfall.parquet'
prep_freq_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/{par_month}_flag_freq.parquet'
prep_feature_360 = f'dbfs:/Volumes/int-cu-siampiwat/staging/raw/{par_month}_360_feature.parquet'
profile_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/profile_chula.csv'
date_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/day_type_apr_june_26.csv'
nantional_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/country_group_chula.csv'
home_region_path = 'dbfs:/Volumes/int-cu-siampiwat/staging/tmp/home_region_chula.csv'

# report path
report_path = f'dbfs:/Volumes/int-cu-siampiwat/staging/report/report3/{par_month}/'

In [0]:
df_date = (spark.read
      .option("header", "true").option("inferSchema", "true")
      .csv(date_path)).select('date','WEEK','day_type_final')
    

In [0]:
df = spark.read.parquet(prep_path)
     # raw footfall by mall & customertype (is_bmr, is_non_bmr, is_foriegner)

df_freq = spark.read.parquet(prep_freq_path) # raw freq by mall
df_360 = spark.read.parquet(prep_feature_360)
     # raw profile
df_merge = df.join(F.broadcast(df_freq), ['msisdn','name'], 'left')\
    .join(F.broadcast(df_360.drop('is_bmr','is_non_bmr','is_foriegner','a_country_name','demo_tourist_sim_v1_tourist_bin','roaming_flag')), ['msisdn'], 'inner')
df_merge = df_merge\
    .join(df_date, [df_merge.par_day == df_date.date], "left")\
    .withColumnRenamed('par_month','month')
display(df_merge.count())

In [0]:
# rename
df_intermediate_time = df_merge.withColumnRenamed("par_hour", "time")

In [0]:
# Frequent Customers
df_intermediate_visit = df_intermediate_time.withColumn('frequent_customers', 
  F.when((F.col('visit_freq_num')>=3), F.lit(1)).otherwise(F.lit(0))
) 
# df_intermediate_visit.limit(5).display()

In [0]:
df_all_columns = df_intermediate_visit.withColumnRenamed('weekly_mall_visitors','weekly_active')
df_all_columns = df_all_columns.fillna(0, 
    subset=[
    'frequent_customers',
    'weekly_active'
    ]
)

In [0]:
df_all_columns = df_all_columns.withColumnRenamed('day_type_final','day_type')\
    .drop('region')\
        .withColumnRenamed('nationality_group','region')\
            .withColumnRenamed('name','mall')
df_all_columns = df_all_columns.withColumn('interest_IS', F.col("interest_IS").cast("string"))
df_all_columns = df_all_columns.withColumn('interest_IL', F.col("interest_IL").cast("string"))
df_all_columns = df_all_columns.withColumn('interest_IF', F.col("interest_IF").cast("string"))
df_all_columns = df_all_columns.withColumn('interest_I', F.col("interest_I").cast("string"))

In [0]:
core_columns = [
    'latitude',
    'longitude',
    'mall',
    'province',	
    'district',
    'sub_district',
    'day_type',
    'date',
    'time',
    'gender',
    'age_range'
]

In [0]:
R3P_BMR_columns = [
    'monthly_pay',
    'place_type' # BMR
]

R3P_Foreigner_columns = [
    'nationality', # foreigner
    'region', # foreigner
    'monthly_pay'
]

R3P_Non_BMR_columns = [
    'monthly_pay'
]
df_bmr_r3p = df_all_columns.filter(F.col('is_bmr')==1)\
    .withColumn("place_type",F.when(F.col("place_type") == "other", F.lit("home_work_unidentified")).otherwise(F.col("place_type")))\
    .groupBy(core_columns+R3P_BMR_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))
df_non_bmr_r3p = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns+R3P_Non_BMR_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))
df_foreigner_r3p = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns+R3P_Foreigner_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))

save_to_csv(df_bmr_r3p, report_path+ f"report3_bmr_r3p_{par_month}.csv")
save_to_csv(df_non_bmr_r3p, report_path+ f"report3_non_bmr_r3p_{par_month}.csv")
save_to_csv(df_foreigner_r3p, report_path+ f"report3_foreigner_r3p_{par_month}.csv")

In [0]:
R3IS_BMR_columns = [
    'interest_IS',
    'monthly_pay'
]

R3IS_Non_BMR_columns = [
    'interest_IS',
    'monthly_pay'
]

R3IS_Foreigner_columns = [
    'interest_IS',
    'monthly_pay',
    'nationality', # foreigner
    'region' # foreigner
]

df_bmr_r3is = df_all_columns.filter(F.col('is_bmr')==1)\
    .groupBy(core_columns+R3IS_BMR_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))
df_non_bmr_r3is = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns+R3IS_Non_BMR_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))
df_foreigner_r3is = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns+R3IS_Foreigner_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))

df_bmr_r3is = df_bmr_r3is.withColumnRenamed('interest_IS','interest')
df_non_bmr_r3is = df_non_bmr_r3is.withColumnRenamed('interest_IS','interest')
df_foreigner_r3is = df_foreigner_r3is.withColumnRenamed('interest_IS','interest')

save_to_csv(df_bmr_r3is, report_path+ f"report3_bmr_r3is_{par_month}.csv")
save_to_csv(df_non_bmr_r3is, report_path+ f"report3_non_bmr_r3is_{par_month}.csv")
save_to_csv(df_foreigner_r3is, report_path+ f"report3_foreigner_r3is_{par_month}.csv")

In [0]:
R3IL_BMR_columns = [
    'interest_IL',
    'monthly_pay',
]

R3IL_Non_BMR_columns = [
    'interest_IL',
    'monthly_pay',
]

R3IL_Foreigner_columns = [
    'interest_IL',
    'monthly_pay',
    'nationality', # foreigner
    'region' # foreigner
]

df_bmr_r3il = df_all_columns.filter(F.col('is_bmr')==1)\
    .groupBy(core_columns+R3IL_BMR_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))
df_non_bmr_r3il = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns+R3IL_Non_BMR_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))
df_foreigner_r3il = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns+R3IL_Foreigner_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))

df_bmr_r3il = df_bmr_r3il.withColumnRenamed('interest_IL','interest')
df_non_bmr_r3il = df_non_bmr_r3il.withColumnRenamed('interest_IL','interest')
df_foreigner_r3il = df_foreigner_r3il.withColumnRenamed('interest_IL','interest')

save_to_csv(df_bmr_r3il, report_path+f"report3_bmr_r3il_{par_month}.csv")
save_to_csv(df_non_bmr_r3il, report_path+f"report3_non_bmr_r3il_{par_month}.csv")
save_to_csv(df_foreigner_r3il, report_path+f"report3_foreigner_r3il_{par_month}.csv")

In [0]:
R3IF_BMR_columns = [
    'interest_IF',
    'monthly_pay'
]

R3IF_Non_BMR_columns = [
    'interest_IF',
    'monthly_pay'
]

R3IF_Foreigner_columns = [
    'interest_IF',
    'monthly_pay',
    'nationality', # foreigner
    'region' # foreigner
]
df_bmr_r3if = df_all_columns.filter(F.col('is_bmr')==1)\
    .groupBy(core_columns+R3IF_BMR_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))
df_non_bmr_r3if = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns+R3IF_Non_BMR_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))
df_foreigner_r3if = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns+R3IF_Foreigner_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))

df_bmr_r3if = df_bmr_r3if.withColumnRenamed('interest_IF','interest')
df_non_bmr_r3if = df_non_bmr_r3if.withColumnRenamed('interest_IF','interest')
df_foreigner_r3if = df_foreigner_r3if.withColumnRenamed('interest_IF','interest')

save_to_csv(df_bmr_r3if, report_path+f"report3_bmr_r3if_{par_month}.csv")
save_to_csv(df_non_bmr_r3if, report_path+f"report3_non_bmr_r3if_{par_month}.csv")
save_to_csv(df_foreigner_r3if, report_path+f"report3_foreigner_r3if_{par_month}.csv")

In [0]:
R3I_BMR_columns = [
    'interest_I',
    'monthly_pay'
]

R3I_Non_BMR_columns = [
    'interest_I',
    'monthly_pay'
]

R3I_Foreigner_columns = [
    'interest_I',
    'monthly_pay',
    'nationality', # foreigner
    'region' # foreigner
]

df_bmr_r3i = df_all_columns.filter(F.col('is_bmr')==1)\
    .groupBy(core_columns+R3I_BMR_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))
df_non_bmr_r3i = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns+R3I_Non_BMR_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))
df_foreigner_r3i = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns+R3I_Foreigner_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))

df_bmr_r3i = df_bmr_r3i.withColumnRenamed('interest_I','interest')
df_non_bmr_r3i = df_non_bmr_r3i.withColumnRenamed('interest_I','interest')
df_foreigner_r3i = df_foreigner_r3i.withColumnRenamed('interest_I','interest')

save_to_csv(df_bmr_r3i, report_path+f"report3_bmr_r3i_{par_month}.csv")
save_to_csv(df_non_bmr_r3i, report_path+f"report3_non_bmr_r3i_{par_month}.csv")
save_to_csv(df_foreigner_r3i, report_path+f"report3_foreigner_r3i_{par_month}.csv")

In [0]:
R3F_BMR_columns = [
    'frequent_customers',
    'weekly_active'
]

R3F_Non_BMR_columns = [
    'frequent_customers',
    'weekly_active'
]

R3F_Foreigner_columns = [
    'nationality', # foreigner
    'region', # foreigner
    'frequent_customers',
    'weekly_active'
]

df_bmr_r3f = df_all_columns.filter(F.col('is_bmr')==1)\
    .groupBy(core_columns+R3F_BMR_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))
df_non_bmr_r3f = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns+R3F_Non_BMR_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))
df_foreigner_r3f = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns+R3F_Foreigner_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))

save_to_csv(df_bmr_r3f, report_path+f"report3_bmr_r3f_{par_month}.csv")
save_to_csv(df_non_bmr_r3f, report_path+f"report3_non_bmr_r3f_{par_month}.csv")
save_to_csv(df_foreigner_r3f, report_path+f"report3_foreigner_r3f_{par_month}.csv")

In [0]:
core_columns_con = ['latitude','longitude','mall','province',	'district','sub_district','day_type','date','time']
df_bmr_r3c_con = df_all_columns.filter(F.col('is_bmr')==1)\
    .groupBy(core_columns_con).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))
df_non_bmr_r3c_con = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns_con).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))
df_foreigner_r3c_con = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns_con).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))

save_to_csv(df_bmr_r3c_con, report_path+ f"report3_bmr_r3c_con_{par_month}.csv")
save_to_csv(df_non_bmr_r3c_con, report_path+ f"report3_non_bmr_r3c_con_{par_month}.csv")
save_to_csv(df_foreigner_r3c_con, report_path+ f"report3_foreigner_r3c_con_{par_month}.csv")

In [0]:
R3F_BMR_demo_columns = []

R3F_Non_demo_con_columns = []

R3F_Foreigner_demo_columns = [
    'nationality', # foreigner
    'region']

df_bmr_r3c_demo = df_all_columns.filter(F.col('is_bmr')==1)\
    .groupBy(core_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))
df_non_bmr_r3c_demo = df_all_columns.filter(F.col('is_non_bmr')==1)\
    .groupBy(core_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))
df_foreigner_r3c_demo = df_all_columns.filter(F.col('is_foriegner')==1)\
    .groupBy(core_columns+R3F_Foreigner_demo_columns).agg(F.count('msisdn').alias('hourly_unique_visitor_cnt'))

save_to_csv(df_bmr_r3c_demo, report_path+ f"report3_bmr_r3c_demo_{par_month}.csv")
save_to_csv(df_non_bmr_r3c_demo, report_path+ f"report3_non_bmr_r3c_demo_{par_month}.csv")
save_to_csv(df_foreigner_r3c_demo, report_path+ f"report3_foreigner_r3c_demo_{par_month}.csv")